# 🔎 Explore your dataset (Week 5 — Data Analytics)
Before training, look at your data. Bad data → bad model, no matter how fancy the model is.

In [ ]:
# 👉 Change this to YOUR GitHub username, then run every cell (Runtime → Run all)
GITHUB_USERNAME = "your-username"

import os
repo = f"https://github.com/{GITHUB_USERNAME}/snapsnipe"
if os.path.exists("snapsnipe"):
    !cd snapsnipe && git pull -q
else:
    !git clone -q {repo}
%cd snapsnipe
!pip install -q -r requirements.txt

## 1. Load every photo into a table

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
from PIL import Image, ImageStat
from config import CATEGORIES

rows = []
for cat in CATEGORIES:
    for path in sorted(Path("data/raw", cat).glob("*")):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        img = Image.open(path)
        gray = img.convert("L")
        rows.append({
            "path": str(path),
            "category": cat,
            "width": img.width,
            "height": img.height,
            "brightness": ImageStat.Stat(gray).mean[0],       # 0 = black, 255 = white
            "file_kb": path.stat().st_size / 1024,
            "hash": hashlib.md5(path.read_bytes()).hexdigest(),  # same hash = exact duplicate
        })

df = pd.DataFrame(rows)
print(f"{len(df)} photos total")
df.head()

## 2. How many photos per category?

In [ ]:
import matplotlib.pyplot as plt

counts = df["category"].value_counts().reindex(CATEGORIES, fill_value=0)
counts.plot(kind="bar", title="Photos per category", color="#4C72B0")
plt.ylabel("# photos"); plt.show()

ratio = counts.max() / max(counts.min(), 1)
print(counts.to_string())
print("✅ Balanced enough" if ratio <= 2 else f"⚠️ Biggest category has {ratio:.1f}× the smallest — add photos to the small ones")

## 3. Image sizes and brightness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cat, group in df.groupby("category"):
    axes[0].scatter(group["width"], group["height"], label=cat, alpha=0.6)
    axes[1].hist(group["brightness"], bins=20, alpha=0.5, label=cat)
axes[0].set(title="Image size", xlabel="width (px)", ylabel="height (px)")
axes[1].set(title="Brightness", xlabel="0 = dark   →   255 = bright")
axes[0].legend(); axes[1].legend(); plt.show()

df.groupby("category")[["width", "height", "brightness", "file_kb"]].mean().round(0)

## 4. Look at a sample of each category
Do they *look* like what they're labeled? Anything surprising?

In [ ]:
n = 6
fig, axes = plt.subplots(len(CATEGORIES), n, figsize=(2 * n, 2 * len(CATEGORIES)), squeeze=False)
for r, cat in enumerate(CATEGORIES):
    sample = df[df.category == cat].sample(min(n, (df.category == cat).sum()), random_state=0)
    for c in range(n):
        ax = axes[r][c]; ax.axis("off")
        if c < len(sample):
            ax.imshow(Image.open(sample.iloc[c]["path"]).convert("RGB"))
        if c == 0:
            ax.set_title(cat, loc="left")
plt.tight_layout(); plt.show()

## 5. Problems to fix
Delete or replace these on GitHub, then re-run the notebook.

In [ ]:
problems = 0

tiny = df[(df.width < 128) | (df.height < 128)]
for p in tiny.path: print("🔍 tiny image:", p); problems += 1

dupes = df[df.duplicated("hash", keep="first")]
for p in dupes.path: print("👯 exact duplicate:", p); problems += 1

dark = df[df.brightness < 30]
bright = df[df.brightness > 230]
for p in dark.path: print("🌑 very dark:", p); problems += 1
for p in bright.path: print("☀️ very bright:", p); problems += 1

print("\n✅ No problems found!" if problems == 0 else f"\n{problems} things to look at")

## 6. Write your insights ✍️
Add 3 things you noticed to `data/README.md` under **Dataset insights**.
Examples: *"Most salad photos are taken from above"*, *"All campus-spot photos are daytime"*, *"Burrito photos are darker on average."*

Why it matters: the model will learn these patterns too — even the ones you didn't mean to teach it!